In [1]:
import polars as pl
import plotly.express as px
import wandb

In [2]:
api = wandb.Api()
artifact = api.artifact("flood-forecasting/flood-dataset-daily:latest")
artifact_dir = artifact.download()

df = pl.read_parquet(f"{artifact_dir}/flood_model_daily.parquet")

wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from C:\Users\zurek\_netrc.
wandb:   1 of 1 files downloaded.  


In [39]:
df_new = df

In [40]:
print(df_new.columns)
print(len(df_new.columns))


['site_id', 'observed_date', 'latitude', 'longitude', 'streamflow_cfs_mean', 'gage_height_ft_mean', 'precipitation_mm', 'temperature_c_mean', 'temperature_c_max', 'temperature_c_min', 'wind_speed_ms_mean', 'specific_humidity_kgkg_mean', 'surface_pressure_pa_mean', 'shortwave_radiation_wm2_mean', 'longwave_radiation_wm2_mean', 'potential_evaporation_mm', 'cape_jkg_mean', 'convective_precip_fraction_mean', 'station_name', 'huc_code', 'drainage_area_sq_km', 'is_reference_hcdn2009', 'elev_mean_m', 'elev_max_m', 'elev_min_m', 'SLOPE_PCT', 'ASPECT_NORTHNESS', 'ASPECT_EASTNESS', 'geology_class_reedbush', 'geology_desc_hunt', 'p_mean', 'pet_mean', 'aridity_index', 'p_seasonality', 'frac_snow', 'high_prec_freq', 'low_prec_freq', 'hydroatlas_elev_m', 'hydroatlas_slope_deg', 'hydroatlas_temp_mean_c', 'hydroatlas_precip_mm_yr', 'hydroatlas_pet_mm_yr', 'hydroatlas_aridity', 'hydroatlas_clay_pct', 'hydroatlas_sand_pct', 'hydroatlas_forest_pct', 'hydroatlas_crop_pct', 'hydroatlas_urban_pct']
48


In [41]:
site_locations = (
    df_new
    .select(["site_id", "station_name", "latitude", "longitude"])
    .unique(subset=["site_id"])
    .drop_nulls(["latitude", "longitude"])
)

In [42]:
fig = px.scatter_mapbox(
    site_locations.to_pandas(),
    lat="latitude",
    lon="longitude",
    hover_name="site_id",
    hover_data=["station_name"],
    zoom=4,
    height=800,
    title="All Sites Map"
)

fig.update_layout(mapbox_style="open-street-map")
fig.show()

C:\Users\zurek\AppData\Local\Temp\ipykernel_17720\2250388807.py:1: DeprecationWarning:

*scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/



In [45]:
cat_unique_counts = (
    df_new
    .select(
        (pl.selectors.string() | pl.selectors.categorical())
            .n_unique()
    )
)

print(cat_unique_counts)

df_new = df_new.with_columns(
    pl.col("station_name")
      .str.strip_chars()
      .str.extract(r"[,\s]([^,\s]+)$")
      .alias("STATE")
)

print(df_new.select("STATE").unique())

shape: (1, 6)
┌─────────┬──────────────┬──────────────────────┬──────────────────────┬───────────────────┬───────┐
│ site_id ┆ station_name ┆ is_reference_hcdn200 ┆ geology_class_reedbu ┆ geology_desc_hunt ┆ STATE │
│ ---     ┆ ---          ┆ 9                    ┆ sh                   ┆ ---               ┆ ---   │
│ u32     ┆ u32          ┆ ---                  ┆ ---                  ┆ u32               ┆ u32   │
│         ┆              ┆ u32                  ┆ u32                  ┆                   ┆       │
╞═════════╪══════════════╪══════════════════════╪══════════════════════╪═══════════════════╪═══════╡
│ 1049    ┆ 1049         ┆ 2                    ┆ 7                    ┆ 22                ┆ 110   │
└─────────┴──────────────┴──────────────────────┴──────────────────────┴───────────────────┴───────┘
shape: (43, 1)
┌───────────────────┐
│ STATE             │
│ ---               │
│ str               │
╞═══════════════════╡
│ Bndry             │
│ M0                │
│ 14)    

In [ ]:
numeric_cols = df_new.select(pl.selectors.numeric()).columns

corr_df = pl.DataFrame({
    col1: [
        df_new.select(pl.corr(col1, col2)).item()
        for col2 in numeric_cols
    ]
    for col1 in numeric_cols
})

corr_df = corr_df.with_columns(
    pl.Series("column", numeric_cols)
).select(["column"] + numeric_cols)
corr_df = corr_df.filter(
    pl.sum_horizontal(pl.all().exclude("column").is_nan()) < len(numeric_cols) - 1
)
valid_vars = corr_df["column"].to_list()
corr_df = corr_df.select(["column"] + valid_vars)

print(corr_df.shape)

(30, 31)


In [36]:
corr_pd = corr_df.to_pandas().set_index("column")

n_vars = len(corr_pd.columns)

fig_size = max(600, n_vars * 25)

fig = px.imshow(
    corr_pd,
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    aspect="auto"
)

fig.update_layout(
    title="Correlation Heatmap",
    width=fig_size,
    height=fig_size,
    xaxis=dict(
        tickangle=45,
        automargin=True
    ),
    yaxis=dict(
        automargin=True
    )
)

fig.update_xaxes(side="bottom")

fig.show()

In [16]:
def pull_wandb(file_name: str,file_path: str = None,n_rows: int | None = None) -> pl.DataFrame:
    run = wandb.init(
        project="flood-forecasting",
        entity="connorjsmith28-rice-university",
        job_type="preprocessing"
    )
    artifact = run.use_artifact(
        f"connorjsmith28-rice-university/flood-forecasting/{file_path}:latest"
    )
    artifact_dir = artifact.download()
    return pl.read_parquet(
        f"{artifact_dir}/{file_name}.parquet",
        n_rows=n_rows,)
config = {"n_rows": 100,
    "file_path": "flood-dataset-missouri",
    "file_name": "flood_model_missouri"
}

In [17]:
df = pull_wandb(config["file_name"],config["file_path"],config['n_rows'])

wandb: Currently logged in as: cz88 (connorjsmith28-rice-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Downloading large artifact 'flood-dataset-missouri:latest', 604.73MB. 1 files...
wandb:   1 of 1 files downloaded.  
Done. 00:00:26.3 (23.0MB/s)


In [19]:
print(df)

shape: (100, 51)
┌──────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ site_id  ┆ observati ┆ latitude  ┆ longitude ┆ … ┆ hydroatla ┆ hydroatla ┆ hydroatla ┆ hydroatla │
│ ---      ┆ on_hour   ┆ ---       ┆ ---       ┆   ┆ s_sand_pc ┆ s_forest_ ┆ s_crop_pc ┆ s_urban_p │
│ str      ┆ ---       ┆ f64       ┆ f64       ┆   ┆ t         ┆ pct       ┆ t         ┆ ct        │
│          ┆ datetime[ ┆           ┆           ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│          ┆ μs, UTC]  ┆           ┆           ┆   ┆ f64       ┆ f64       ┆ f64       ┆ f64       │
╞══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 06923250 ┆ 2008-08-0 ┆ 37.684306 ┆ -92.92463 ┆ … ┆ 34.502088 ┆ 49.937232 ┆ 36.317987 ┆ 0.90795   │
│          ┆ 3         ┆           ┆ 9         ┆   ┆           ┆           ┆           ┆           │
│          ┆ 21:00:00  ┆           ┆           ┆   ┆           ┆          